# Part 2: Simple Logistic Regression Model
We will be using a simple bag-of-words representation for our initial, simple model. To train this model, we will be using the pytorch library.

In [1]:
from modelling import BagofWordsDataset, train
from torch.utils.data import DataLoader
from pathlib import Path
import torch.nn as nn
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
np.random.seed(SEED)

In [2]:
from typing import Tuple

DATASET_DIRECTORY = Path("data/final_dataset")
VOCABULARY = DATASET_DIRECTORY / "top10k_vocabulary.csv"
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
EPOCHS = 10
L2_LAMBDA = 0.01 # add l2 reg to discourage the model from giving giant weights to extremely rare tokens

# To stop using domain: remove "include_domain=True and unique_domains=..." for both BagOfWordsDatasets
def get_dataloaders(use_domain : bool) -> Tuple[int, DataLoader, DataLoader]:
    train_ds = BagofWordsDataset(
        DATASET_DIRECTORY / "train.csv",
        VOCABULARY, 
        include_domain=use_domain)
    train_dl = DataLoader(
        train_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=True)
    val_dl = DataLoader(
        BagofWordsDataset(
            DATASET_DIRECTORY / "val.csv",
            VOCABULARY, 
            include_domain=use_domain,
            unique_domains=train_ds.__get_unique_domains__() if use_domain else None), 
        batch_size=BATCH_SIZE, 
        shuffle=False)
    return len(train_ds[0][0]), train_dl, val_dl

vocab_size, train_dl, val_dl = get_dataloaders(False)

tokenizing 634696 documents


100%|██████████| 634696/634696 [04:23<00:00, 2408.96it/s]


tokenizing 79337 documents


100%|██████████| 79337/79337 [00:33<00:00, 2368.09it/s]


In [3]:
from modelling.logistic_regression import LogisticRegression
model = LogisticRegression(vocab_size)
loss = nn.BCELoss()

train(model, loss, train_dl, val_dl, LEARNING_RATE, EPOCHS)

100%|██████████| 1240/1240 [00:02<00:00, 565.46it/s]


Epochs: 1 | Train Loss:  0.449 | Val Loss:  0.405 
[+] Saved checkpoint 'best_model_2026-03-15_11:57_epoch_0.pth'


100%|██████████| 1240/1240 [00:02<00:00, 564.82it/s]


Epochs: 2 | Train Loss:  0.382 | Val Loss:  0.382 
[+] Saved checkpoint 'best_model_2026-03-15_11:57_epoch_1.pth'


100%|██████████| 1240/1240 [00:02<00:00, 556.37it/s]


...

Epochs: 5 | Train Loss:  0.353 | Val Loss:  0.370 
[+] Saved checkpoint 'best_model_2026-03-15_11:57_epoch_4.pth'


100%|██████████| 1240/1240 [00:02<00:00, 559.39it/s]


Epochs: 6 | Train Loss:  0.349 | Val Loss:  0.368 
[+] Saved checkpoint 'best_model_2026-03-15_11:57_epoch_5.pth'


100%|██████████| 1240/1240 [00:02<00:00, 559.85it/s]

Epochs: 7 | Train Loss:  0.347 | Val Loss:  0.369 
Early stopping


In [4]:
from sklearn.metrics import f1_score

@torch.no_grad()
def evaluate_f1(model : nn.Module, dl : DataLoader) -> float:
    all_predictions = []
    all_labels = []

    for inputs, labels in dl:
        y_probs = model(inputs)
        y_pred = (y_probs >= 0.5).long()
        all_predictions.append(y_pred)
        all_labels.append(labels)
    
    all_predictions = torch.cat(all_predictions)
    all_labels = torch.cat(all_labels)

    return f1_score(all_labels, all_predictions)

print("F1 score of basic logistic regression: ", evaluate_f1(model, val_dl))

F1 score of basic logistic regression:  0.8878901092727324


In [5]:
MODELS_DIRECTORY = Path("data/models")
torch.save(model.state_dict(), MODELS_DIRECTORY / "basic_log_reg.pt")

## 2.1. Including content
As we saw in the data analysis section, the label of an article is strongly controlled by the domain it originates from, so including this information should surely improve the model

In [6]:
vocab_size, train_dl, val_dl = get_dataloaders(True)

tokenizing 634696 documents


100%|██████████| 634696/634696 [04:27<00:00, 2373.36it/s]


tokenizing 79337 documents


100%|██████████| 79337/79337 [00:33<00:00, 2334.19it/s]


In [7]:
model = LogisticRegression(vocab_size)
loss = nn.BCELoss()

train(model, loss, train_dl, val_dl, LEARNING_RATE, EPOCHS)

100%|██████████| 1240/1240 [00:05<00:00, 226.46it/s]


Epochs: 1 | Train Loss:  0.376 | Val Loss:  0.276 
[+] Saved checkpoint 'best_model_2026-03-15_12:06_epoch_0.pth'


100%|██████████| 1240/1240 [00:05<00:00, 228.26it/s]


Epochs: 2 | Train Loss:  0.223 | Val Loss:  0.186 
[+] Saved checkpoint 'best_model_2026-03-15_12:06_epoch_1.pth'


100%|██████████| 1240/1240 [00:05<00:00, 228.38it/s]


...

Epochs: 8 | Train Loss:  0.040 | Val Loss:  0.041 
[+] Saved checkpoint 'best_model_2026-03-15_12:06_epoch_7.pth'


100%|██████████| 1240/1240 [00:05<00:00, 228.08it/s]


Epochs: 9 | Train Loss:  0.034 | Val Loss:  0.036 
[+] Saved checkpoint 'best_model_2026-03-15_12:06_epoch_8.pth'


100%|██████████| 1240/1240 [00:05<00:00, 227.96it/s]

Epochs: 10 | Train Loss:  0.028 | Val Loss:  0.031 
[+] Saved checkpoint 'best_model_2026-03-15_12:06_epoch_9.pth'


In [8]:
print("F1 score of logistic regression, including domain: ", evaluate_f1(model, val_dl))

F1 score of logistic regression, including domain:  0.9968016532992177
